In [1]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# data load
X =np.load('data/f4/initial_inputs.npy')
y=np.load('data/f4/initial_outputs.npy')

In [3]:
# Data Normalization
y_mean_val = np.mean(y)
y_std_val  = np.std(y)
y_norm    = (y - y_mean_val) / y_std_val if y_std_val > 0 else y

In [4]:
# The best data
best_f = np.max(y_norm)
best_idx = np.argmax(y_norm)
print(f"The best data is: X = {X[best_idx]}, y_norm = {best_f:.4f}")

The best data is: X = [0.57776561 0.42877174 0.42582587 0.24900741], y_norm = 1.8827


In [5]:
# GP model: kernel n (dim) dimension
dim=4
noise_assumption = 1e-6
rbf_lengthscale  = np.ones(dim) * 1
kernel = RBF(length_scale=1, length_scale_bounds='fixed')
model  = GaussianProcessRegressor(kernel=kernel, alpha=noise_assumption)
model.fit(X, y_norm)

,kernel,RBF(length_scale=1)
,alpha,1e-06
,optimizer,'fmin_l_bfgs_b'
,n_restarts_optimizer,0
,normalize_y,False
,copy_X_train,True
,n_targets,None
,random_state,None
,kernel__length_scale,1
,kernel__length_scale_bounds,'fixed'


In [6]:
# Grid 3D for prediction
from scipy.stats import qmc

n_grid = 5000 
sampler = qmc.LatinHypercube(d=dim, seed=42)
sample = sampler.random(n=n_grid)

# Compute bounds
l_bounds = X.min(axis=0)  
u_bounds = X.max(axis=0)  
x_grid_d = qmc.scale(sample, l_bounds, u_bounds)

In [7]:
# GP prediction ---
post_mean, post_std = model.predict(x_grid_d, return_std=True)
#print(post_mean)
#print(post_std)

In [8]:
###  New acquisition Function EI (same GP)

In [9]:
#EI (acquisition function definition)
def expected_improvement(mean, std, best_f, xi=0.01):
    Z  = (mean - best_f - xi) / (std + 1e-9)
    ei = (mean - best_f - xi) * norm.cdf(Z) + std * norm.pdf(Z)
    ei[std <= 0.0] = 0.0
    return ei

ei_values = expected_improvement(post_mean, post_std, best_f, xi=0.01)
#print (ei_values)

In [10]:
# Calculate the maximum value
idx_next  = np.argmax(ei_values)# simple grid
x_next    = x_grid_d[idx_next]
std_next = post_std[idx_next]
y_next = post_mean[idx_next]

print(f"proposed query(EI) (EI): x1={x_next[0]:.8f}, x2={x_next[1]:.8f}, x3={x_next[2]:.8f}, x4={x_next[3]:.8f}")
print(f"proposed query std (EI): {std_next:.6f}")
print(f"proposed query y_next (EI): {y_next:.6f}")

proposed query(EI) (EI): x1=0.48334308, x2=0.34241170, x3=0.32049846, x4=0.44028479
proposed query std (EI): 0.013867
proposed query y_next (EI): 2.259526


In [11]:
   ## New acquisition Function UCB (same GP)

In [12]:
#UCB
k = 1.5 #Initial exploration
acquisition = post_mean + k * post_std
idx_next = np.argmax(acquisition)
x_next = x_grid_d[idx_next]
std_next = post_std[idx_next]
y_next = post_mean[idx_next]
# The next point
idx_next  = np.argmax(ei_values)# simple grid
x_next    = x_grid_d[idx_next]
print(f"Proposed query (UCB): x1={x_next[0]:.8f}, x2={x_next[1]:.8f}, x3={x_next[2]:.8f}, x4={x_next[3]:.8f}")
print(f"proposed query std (UCB): {std_next:.6f}")
print(f"proposed query y_next (UCB): {y_next:.6f}")

Proposed query (UCB): x1=0.48334308, x2=0.34241170, x3=0.32049846, x4=0.44028479
proposed query std (UCB): 0.013867
proposed query y_next (UCB): 2.259526


In [13]:
# Write output file
import os
folder = 'results'
filename = 'output_F4.txt'
file_path = os.path.join(folder, filename)
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(f"{x_next[0]:.6f} {x_next[1]:.6f} {x_next[2]:.6f} {x_next[3]:.6f}\n")